# 10 — Teacher Labeling Pipeline (GPT-4o → Pseudo-Labels)

Use the fine-tuned GPT-4o model as a **teacher** to label large-scale unlabeled face
datasets with the 5-signal scoring schema. These pseudo-labels then train lightweight
ONNX student models (knowledge distillation).

## Strategy
1. Download unlabeled face datasets (FFHQ, UTKFace, CelebA-HQ)
2. Batch-label via `ft:gpt-4o-2024-08-06:personal:radianceiq-skin:DHBaOo20`
3. Quality-filter: reject low-confidence, detect outliers
4. Consistency check: re-label 5% subset, discard high-variance images
5. Save unified annotation format for student training

## Output Schema (per image)
```json
{
  "image_id": "ffhq_00001",
  "source": "ffhq",
  "signals": {
    "structure": 72, "hydration": 65, "inflammation": 88,
    "sunDamage": 79, "elasticity": 70
  },
  "confidence": "high",
  "conditions": [...],
  "zone_severity": {...},
  "teacher_model": "ft:gpt-4o-2024-08-06:personal:radianceiq-skin:DHBaOo20",
  "labeled_at": "2026-03-27T..."
}
```

## Cost Estimate
- ~$0.01-0.03 per image (GPT-4o with vision, 512px detail)
- 10k images ≈ $100-300
- 50k images ≈ $500-1500

In [ ]:
# Install dependencies (Colab)
!pip install -q openai pillow tqdm gdown requests pandas scikit-learn matplotlib scipy

In [ ]:
import os
import json
import time
import base64
import hashlib
import re
from datetime import datetime, timezone
from pathlib import Path
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from openai import OpenAI, RateLimitError

# ── Config ──────────────────────────────────────────────────────────
# Set via Colab secrets or environment
try:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    BASE_DIR = Path('/content/drive/MyDrive/Glowlytics')
except Exception:
    OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')
    BASE_DIR = Path('./data')

client = OpenAI(api_key=OPENAI_API_KEY)

TEACHER_MODEL = 'ft:gpt-4o-2024-08-06:personal:radianceiq-skin:DHBaOo20'
RAW_DIR = BASE_DIR / 'datasets' / 'raw'
LABELS_DIR = BASE_DIR / 'datasets' / 'teacher_labels'
RAW_DIR.mkdir(parents=True, exist_ok=True)
LABELS_DIR.mkdir(parents=True, exist_ok=True)

# Labeling config
MAX_IMAGES = 10_000          # Total images to label (adjust for budget)
IMAGE_SIZE = 512              # Resize before sending (cost vs quality)
DETAIL_LEVEL = 'low'         # 'low' (85 tokens, $0.005) or 'high' (up to 1k tokens, ~$0.03)
MAX_WORKERS = 5               # Parallel API calls (respect rate limits)
RETRY_LIMIT = 3
RETRY_BACKOFF = 2.0

print(f'Teacher model: {TEACHER_MODEL}')
print(f'Output dir: {LABELS_DIR}')
print(f'Target: {MAX_IMAGES} images at detail={DETAIL_LEVEL}')

## 1. Download Face Datasets

We use three complementary sources:
- **FFHQ** (Flickr-Faces-HQ): 70k high-quality 1024x1024 faces, diverse demographics
- **UTKFace**: 20k+ faces with age/gender/ethnicity labels (useful for elasticity validation)
- **CelebA-HQ**: 30k celebrity faces

For cost efficiency, we sample a balanced subset across age groups and skin tones.

In [ ]:
import gdown
import zipfile
import tarfile
import requests


def download_ffhq_thumbnails(output_dir: Path, max_images: int = 5000):
    """Download FFHQ 128x128 thumbnails (small, fast, sufficient for 224px training).

    Uses the official FFHQ thumbnails128x128 from Google Drive.
    """
    ffhq_dir = output_dir / 'ffhq'
    if ffhq_dir.exists() and len(list(ffhq_dir.glob('*.png'))) > 100:
        existing = len(list(ffhq_dir.glob('*.png')))
        print(f'FFHQ already downloaded: {existing} images')
        return ffhq_dir

    ffhq_dir.mkdir(parents=True, exist_ok=True)

    # FFHQ thumbnails are split into 70 zip files (1k images each)
    # Download first N zips to reach max_images
    n_zips = min(70, (max_images // 1000) + 1)
    print(f'Downloading {n_zips} FFHQ thumbnail zips (~{n_zips}k images)...')

    for i in tqdm(range(n_zips), desc='FFHQ zips'):
        zip_name = f'thumbnails128x128-{i:03d}.zip'
        # These are available from the FFHQ dataset page
        # Using the official Google Drive links from NVIDIA
        zip_path = ffhq_dir / zip_name
        if zip_path.exists():
            continue
        try:
            # FFHQ thumbnails: use the NVIDIA dataset download script or manual GDrive links
            # See https://github.com/NVlabs/ffhq-dataset for official download instructions
            # For automated download, use the tfrecords or the AWS mirror:
            url = f'https://s3.amazonaws.com/research-data-public/ffhq/thumbnails128x128/thumbnails128x128-{i:04d}.zip'
            gdown.download(url, str(zip_path), quiet=True)
            with zipfile.ZipFile(zip_path, 'r') as zf:
                zf.extractall(ffhq_dir)
            zip_path.unlink()  # Clean up zip
        except Exception as e:
            print(f'  Zip {i} failed: {e}')

    count = len(list(ffhq_dir.rglob('*.png')))
    print(f'FFHQ ready: {count} images')
    return ffhq_dir


def download_utkface(output_dir: Path):
    """Download UTKFace dataset (age-labeled faces)."""
    utk_dir = output_dir / 'utkface'
    if utk_dir.exists() and len(list(utk_dir.glob('*.jpg'))) > 100:
        existing = len(list(utk_dir.glob('*.jpg')))
        print(f'UTKFace already downloaded: {existing} images')
        return utk_dir

    utk_dir.mkdir(parents=True, exist_ok=True)
    print('Downloading UTKFace...')

    # UTKFace is available from multiple mirrors
    urls = [
        'https://drive.google.com/uc?id=0BxYys69jI14kYVM3aVhKS1VhRUk',
    ]
    for url in urls:
        try:
            tar_path = utk_dir / 'utkface.tar.gz'
            gdown.download(url, str(tar_path), quiet=False)
            with tarfile.open(tar_path, 'r:gz') as tf:
                tf.extractall(utk_dir)
            tar_path.unlink()
            break
        except Exception as e:
            print(f'  Download failed: {e}')

    count = len(list(utk_dir.rglob('*.jpg')))
    print(f'UTKFace ready: {count} images')
    return utk_dir


def collect_image_paths(raw_dir: Path, max_total: int) -> list[dict]:
    """Collect image paths from all dataset directories with source metadata.

    Returns balanced sample across sources. UTKFace filenames encode
    age_gender_race_date.jpg — we parse age for stratified sampling.
    """
    all_images = []

    # FFHQ
    ffhq_dir = raw_dir / 'ffhq'
    if ffhq_dir.exists():
        for p in sorted(ffhq_dir.rglob('*.png')):
            all_images.append({'path': str(p), 'source': 'ffhq', 'age': None})

    # UTKFace — parse age from filename
    utk_dir = raw_dir / 'utkface'
    if utk_dir.exists():
        for p in sorted(utk_dir.rglob('*.jpg')):
            parts = p.stem.split('_')
            age = int(parts[0]) if parts[0].isdigit() else None
            all_images.append({'path': str(p), 'source': 'utkface', 'age': age})

    # CelebA-HQ (if available)
    celeba_dir = raw_dir / 'celeba_hq'
    if celeba_dir.exists():
        for p in sorted(celeba_dir.rglob('*.jpg')):
            all_images.append({'path': str(p), 'source': 'celeba_hq', 'age': None})

    print(f'Total images found: {len(all_images)}')

    # Stratified sampling: ensure age diversity from UTKFace
    utk_with_age = [img for img in all_images if img['age'] is not None]
    others = [img for img in all_images if img['age'] is None]

    if utk_with_age:
        # Sample across age buckets: 15-25, 25-35, 35-50, 50-65, 65+
        age_buckets = [(15, 25), (25, 35), (35, 50), (50, 65), (65, 100)]
        utk_per_bucket = max_total // (len(age_buckets) * 3)  # ~1/3 from UTKFace
        sampled_utk = []
        for lo, hi in age_buckets:
            bucket = [img for img in utk_with_age if lo <= (img['age'] or 0) < hi]
            np.random.shuffle(bucket)
            sampled_utk.extend(bucket[:utk_per_bucket])
        utk_count = len(sampled_utk)
    else:
        sampled_utk = []
        utk_count = 0

    # Fill remaining budget from other sources
    remaining = max_total - utk_count
    np.random.shuffle(others)
    sampled = sampled_utk + others[:remaining]
    np.random.shuffle(sampled)

    print(f'Sampled {len(sampled)} images (UTKFace: {utk_count}, other: {len(sampled) - utk_count})')
    return sampled[:max_total]


# Download datasets
ffhq_dir = download_ffhq_thumbnails(RAW_DIR, max_images=5000)
utk_dir = download_utkface(RAW_DIR)

# Collect and sample
image_manifest = collect_image_paths(RAW_DIR, MAX_IMAGES)
print(f'\nFinal manifest: {len(image_manifest)} images ready for labeling')

## 2. Teacher Labeling Prompt

Matches the exact prompt from `backend/app.js` but stripped of user context
(no goal, no scan count, no sunscreen) since these are unlabeled dataset images.

In [ ]:
TEACHER_SYSTEM_PROMPT = """You are a dermatology analysis assistant. Analyze the provided facial skin photo and return structured scores.

Score each of the 5 skin health signals 0-100 where 100 = optimal health and 0 = severe concern:
- structure: skin texture quality, pore visibility, surface smoothness, collagen integrity
- hydration: moisture levels, barrier function, dewy vs matte appearance, fine dehydration lines
- inflammation: redness, irritation, active breakouts, pustules, papules, erythema
- sunDamage: hyperpigmentation, sunspots, melasma, UV damage signs, uneven pigmentation
- elasticity: firmness, fine lines, wrinkles, skin laxity, bounce-back quality

Provide per-zone severity assessment:
zone_severity: for each facial zone, rate the dominant concern and severity (0-100).
Zones: forehead, left_cheek, right_cheek, nose, chin, jaw

Identify skin conditions with facial zones:
conditions: [{name, severity ("mild"|"moderate"|"severe"),
  zones: [{region, severity}], description}]

Conditions to check: acne, hyperpigmentation, fine_lines, rosacea,
dehydration, sun_spots, texture_irregularity, dark_circles, enlarged_pores

Also provide:
- confidence: "low", "med", or "high" based on image quality and clarity

Return ONLY valid JSON matching this schema:
{
  "signal_scores": {"structure": number, "hydration": number, "inflammation": number, "sunDamage": number, "elasticity": number},
  "confidence": "low" | "med" | "high",
  "zone_severity": {"forehead": {"dominant_signal": string, "severity": number}, "left_cheek": {...}, "right_cheek": {...}, "nose": {...}, "chin": {...}, "jaw": {...}},
  "conditions": [{"name": string, "severity": "mild"|"moderate"|"severe", "zones": [{"region": string, "severity": "mild"|"moderate"|"severe"}], "description": string}]
}"""

print(f'System prompt length: {len(TEACHER_SYSTEM_PROMPT)} chars')

## 3. Labeling Engine

Batch-labels images with rate limiting, retries, and checkpoint/resume support.
Labels are saved incrementally so a crash doesn't lose progress.

In [ ]:
def image_to_base64(path: str, max_size: int = IMAGE_SIZE) -> str:
    """Load, resize, and base64-encode an image."""
    img = Image.open(path).convert('RGB')
    if max(img.size) > max_size:
        img.thumbnail((max_size, max_size), Image.LANCZOS)
    buf = BytesIO()
    img.save(buf, format='JPEG', quality=85)
    return base64.b64encode(buf.getvalue()).decode('utf-8')


def parse_teacher_response(text: str) -> dict | None:
    """Extract and validate JSON from GPT-4o response."""
    # Strip markdown code fences if present
    text = re.sub(r'^```(?:json)?\s*', '', text.strip())
    text = re.sub(r'\s*```$', '', text.strip())

    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        # Try to find JSON object in response
        match = re.search(r'\{[\s\S]*\}', text)
        if not match:
            return None
        try:
            data = json.loads(match.group())
        except json.JSONDecodeError:
            return None

    # Validate required fields
    scores = data.get('signal_scores')
    if not scores:
        return None

    required_signals = ['structure', 'hydration', 'inflammation', 'sunDamage', 'elasticity']
    for sig in required_signals:
        val = scores.get(sig)
        if not isinstance(val, (int, float)):
            return None
        # Clamp to valid range
        scores[sig] = max(0, min(100, round(val)))

    data['signal_scores'] = scores
    return data


def label_single_image(image_info: dict) -> dict | None:
    """Label a single image with the teacher model. Returns annotation or None."""
    path = image_info['path']
    image_id = hashlib.md5(path.encode()).hexdigest()[:12]

    for attempt in range(RETRY_LIMIT):
        try:
            img_b64 = image_to_base64(path)
            response = client.chat.completions.create(
                model=TEACHER_MODEL,
                messages=[
                    {'role': 'system', 'content': TEACHER_SYSTEM_PROMPT},
                    {'role': 'user', 'content': [
                        {'type': 'text', 'text': 'Analyze this skin photo.'},
                        {'type': 'image_url', 'image_url': {
                            'url': f'data:image/jpeg;base64,{img_b64}',
                            'detail': DETAIL_LEVEL,
                        }},
                    ]},
                ],
                max_tokens=1024,
                temperature=0.1,  # Low temp for consistency
            )

            parsed = parse_teacher_response(response.choices[0].message.content)
            if parsed is None:
                continue

            usage = response.usage
            return {
                'image_id': image_id,
                'image_path': path,
                'source': image_info['source'],
                'age': image_info.get('age'),
                'signals': parsed['signal_scores'],
                'confidence': parsed.get('confidence', 'med'),
                'conditions': parsed.get('conditions', []),
                'zone_severity': parsed.get('zone_severity', {}),
                'teacher_model': TEACHER_MODEL,
                'labeled_at': datetime.now(timezone.utc).isoformat(),
                'tokens': {
                    'prompt': usage.prompt_tokens if usage else 0,
                    'completion': usage.completion_tokens if usage else 0,
                },
            }

        except RateLimitError:
            wait = RETRY_BACKOFF ** (attempt + 1)
            time.sleep(wait)
        except Exception as e:
            if attempt == RETRY_LIMIT - 1:
                print(f'  Failed {image_id}: {e}')
            time.sleep(1)

    return None


print('Labeling engine ready.')

In [ ]:
def load_checkpoint(labels_dir: Path) -> set:
    """Load already-labeled image paths from checkpoint files."""
    labeled = set()
    for shard in sorted(labels_dir.glob('labels_shard_*.jsonl')):
        with open(shard) as f:
            for line in f:
                try:
                    entry = json.loads(line)
                    labeled.add(entry['image_path'])
                except (json.JSONDecodeError, KeyError):
                    pass
    return labeled


def run_labeling(manifest: list[dict], labels_dir: Path, max_workers: int = MAX_WORKERS):
    """Label all images in manifest with checkpointing.

    Saves results in JSONL shards (1000 labels per file) for resilience.
    Skips already-labeled images on resume.
    """
    already_done = load_checkpoint(labels_dir)
    remaining = [img for img in manifest if img['path'] not in already_done]
    print(f'Already labeled: {len(already_done)}, remaining: {len(remaining)}')

    if not remaining:
        print('All images already labeled!')
        return

    shard_idx = len(list(labels_dir.glob('labels_shard_*.jsonl')))
    shard_size = 1000
    shard_buffer = []
    stats = {'success': 0, 'failed': 0, 'total_prompt_tokens': 0, 'total_completion_tokens': 0}

    def flush_shard():
        nonlocal shard_idx, shard_buffer
        if not shard_buffer:
            return
        shard_path = labels_dir / f'labels_shard_{shard_idx:04d}.jsonl'
        with open(shard_path, 'w') as f:
            for entry in shard_buffer:
                f.write(json.dumps(entry) + '\n')
        print(f'  Saved shard {shard_idx}: {len(shard_buffer)} labels → {shard_path.name}')
        shard_idx += 1
        shard_buffer = []

    pbar = tqdm(total=len(remaining), desc='Labeling')

    # Sequential with rate-limit awareness (safer than parallel for vision API)
    for img_info in remaining:
        result = label_single_image(img_info)
        if result:
            shard_buffer.append(result)
            stats['success'] += 1
            stats['total_prompt_tokens'] += result['tokens']['prompt']
            stats['total_completion_tokens'] += result['tokens']['completion']
        else:
            stats['failed'] += 1

        pbar.update(1)

        # Flush shard periodically
        if len(shard_buffer) >= shard_size:
            flush_shard()

        # Progress report every 500 images
        total = stats['success'] + stats['failed']
        if total % 500 == 0 and total > 0:
            est_cost = (stats['total_prompt_tokens'] * 2.5 / 1e6) + (stats['total_completion_tokens'] * 10 / 1e6)
            print(f'  Progress: {total}/{len(remaining)} | '
                  f'Success: {stats["success"]} | Failed: {stats["failed"]} | '
                  f'Est. cost: ${est_cost:.2f}')

    pbar.close()
    flush_shard()  # Final flush

    # Summary
    est_cost = (stats['total_prompt_tokens'] * 2.5 / 1e6) + (stats['total_completion_tokens'] * 10 / 1e6)
    print(f'\n{"="*60}')
    print(f'Labeling complete!')
    print(f'  Success: {stats["success"]}')
    print(f'  Failed:  {stats["failed"]}')
    print(f'  Total tokens: {stats["total_prompt_tokens"] + stats["total_completion_tokens"]:,}')
    print(f'  Estimated cost: ${est_cost:.2f}')
    print(f'{"="*60}')

    return stats


# Run labeling (uncomment when ready — this costs real money)
# labeling_stats = run_labeling(image_manifest, LABELS_DIR)

## 4. Quality Filtering

Remove low-quality labels:
1. Drop `confidence: "low"` (image too blurry/dark for reliable scoring)
2. Drop outliers where any signal is at 0 or 100 (likely parsing errors)
3. Drop images where all 5 signals are within 5 points of each other (no discrimination)

In [ ]:
def load_all_labels(labels_dir: Path) -> list[dict]:
    """Load all label shards into a single list."""
    all_labels = []
    for shard in sorted(labels_dir.glob('labels_shard_*.jsonl')):
        with open(shard) as f:
            for line in f:
                try:
                    all_labels.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
    return all_labels


def quality_filter(labels: list[dict]) -> list[dict]:
    """Filter labels for quality."""
    original_count = len(labels)
    filtered = []
    reject_reasons = {'low_confidence': 0, 'extreme_scores': 0, 'no_discrimination': 0}

    for label in labels:
        signals = label['signals']
        values = list(signals.values())

        # 1. Drop low confidence
        if label.get('confidence') == 'low':
            reject_reasons['low_confidence'] += 1
            continue

        # 2. Drop extreme scores (all 0 or all 100 = likely bad parse)
        if all(v <= 5 for v in values) or all(v >= 95 for v in values):
            reject_reasons['extreme_scores'] += 1
            continue

        # 3. Drop non-discriminating (all within 5pt range = model didn't differentiate)
        if max(values) - min(values) < 5:
            reject_reasons['no_discrimination'] += 1
            continue

        filtered.append(label)

    print(f'Quality filter: {original_count} → {len(filtered)} labels')
    print(f'  Rejected: {json.dumps(reject_reasons)}')
    return filtered


# Load and filter
all_labels = load_all_labels(LABELS_DIR)
if all_labels:
    filtered_labels = quality_filter(all_labels)
else:
    print('No labels found yet. Run the labeling step first.')
    filtered_labels = []

## 5. Consistency Check

Re-label a random 5% of images and measure teacher self-agreement.
Discard images where the teacher disagrees with itself (high variance).
This removes ambiguous images that would inject noise into student training.

In [ ]:
def consistency_check(labels: list[dict], sample_pct: float = 0.05,
                      max_variance: float = 100.0) -> list[dict]:
    """Re-label a subset and discard high-variance images.

    max_variance: maximum allowed mean variance across signals (on 0-100 scale).
    Default 100 = sum of variances across 5 signals can be at most 100,
    i.e., avg per-signal std of ~4.5 points.
    """
    n_check = max(10, int(len(labels) * sample_pct))
    check_indices = np.random.choice(len(labels), size=n_check, replace=False)
    check_subset = [labels[i] for i in check_indices]

    print(f'Consistency check: re-labeling {n_check} images...')
    high_variance_ids = set()
    variances = []

    for label in tqdm(check_subset, desc='Consistency'):
        img_info = {'path': label['image_path'], 'source': label['source']}
        relabel = label_single_image(img_info)

        if relabel is None:
            continue

        # Compute per-signal variance between original and re-label
        total_var = 0
        for sig in ['structure', 'hydration', 'inflammation', 'sunDamage', 'elasticity']:
            v1 = label['signals'][sig]
            v2 = relabel['signals'][sig]
            total_var += (v1 - v2) ** 2

        variances.append(total_var)
        if total_var > max_variance:
            high_variance_ids.add(label['image_id'])

    if variances:
        mean_var = np.mean(variances)
        median_var = np.median(variances)
        pct_rejected = len(high_variance_ids) / len(check_subset) * 100
        print(f'  Mean total variance: {mean_var:.1f}')
        print(f'  Median total variance: {median_var:.1f}')
        print(f'  High-variance (>{max_variance}): {len(high_variance_ids)} ({pct_rejected:.1f}%)')
        print(f'  Avg per-signal std: {np.sqrt(mean_var / 5):.1f} points')

    # Remove high-variance images from full dataset
    if high_variance_ids:
        # Scale rejection rate to full dataset
        # We only checked a subset, so estimate and remove flagged ones
        cleaned = [l for l in labels if l['image_id'] not in high_variance_ids]
        print(f'  Removed {len(labels) - len(cleaned)} high-variance labels')
        return cleaned

    return labels


# Run consistency check (uncomment when labels exist)
# filtered_labels = consistency_check(filtered_labels)

## 6. Train/Val/Test Split & Save

Stratified split ensuring age and signal score diversity in each partition.
Save in the unified format consumed by student training notebooks.

In [ ]:
from sklearn.model_selection import train_test_split


def compute_split_stats(labels: list[dict], split_name: str):
    """Print distribution statistics for a split."""
    signals = {sig: [] for sig in ['structure', 'hydration', 'inflammation', 'sunDamage', 'elasticity']}
    sources = {}
    for label in labels:
        for sig, val in label['signals'].items():
            signals[sig].append(val)
        src = label['source']
        sources[src] = sources.get(src, 0) + 1

    print(f'\n  {split_name} ({len(labels)} images):')
    print(f'    Sources: {sources}')
    for sig, vals in signals.items():
        arr = np.array(vals)
        print(f'    {sig:>14s}: mean={arr.mean():.1f}, std={arr.std():.1f}, '
              f'range=[{arr.min():.0f}, {arr.max():.0f}]')


def save_splits(labels: list[dict], output_dir: Path):
    """Split into train/val/test (80/10/10) and save."""
    # Stratify by overall skin health (mean of all signals)
    mean_scores = [np.mean(list(l['signals'].values())) for l in labels]
    # Bin into quartiles for stratification
    bins = np.quantile(mean_scores, [0.25, 0.5, 0.75])
    strata = np.digitize(mean_scores, bins)

    # Fallback to non-stratified split if strata are degenerate
    n_unique = len(set(strata))
    use_stratify = n_unique >= 2 and all(
        np.sum(strata == v) >= 2 for v in set(strata)
    )

    train, temp, strata_train, strata_temp = train_test_split(
        labels, strata, test_size=0.2, random_state=42,
        stratify=strata if use_stratify else None,
    )
    val, test = train_test_split(
        temp, test_size=0.5, random_state=42,
        stratify=strata_temp if use_stratify else None,
    )

    splits = {'train': train, 'val': val, 'test': test}

    print(f'Split: train={len(train)}, val={len(val)}, test={len(test)}')
    for name, data in splits.items():
        compute_split_stats(data, name)
        split_path = output_dir / f'{name}.json'
        with open(split_path, 'w') as f:
            json.dump(data, f, indent=2)
        print(f'  Saved {split_path}')

    # Also save a compact version for quick loading
    compact = {
        'metadata': {
            'teacher_model': TEACHER_MODEL,
            'total_images': len(labels),
            'created_at': datetime.now(timezone.utc).isoformat(),
            'splits': {name: len(data) for name, data in splits.items()},
        },
        'signal_stats': {},
    }
    for sig in ['structure', 'hydration', 'inflammation', 'sunDamage', 'elasticity']:
        vals = [l['signals'][sig] for l in labels]
        compact['signal_stats'][sig] = {
            'mean': float(np.mean(vals)),
            'std': float(np.std(vals)),
            'min': float(np.min(vals)),
            'max': float(np.max(vals)),
        }
    with open(output_dir / 'dataset_info.json', 'w') as f:
        json.dump(compact, f, indent=2)

    return splits


if filtered_labels:
    splits = save_splits(filtered_labels, LABELS_DIR)
else:
    print('No filtered labels to split. Run labeling and filtering first.')

## 7. Label Distribution Visualization

In [ ]:
import matplotlib.pyplot as plt


SIGNAL_COLORS = {
    'structure': '#7DE7E1',
    'hydration': '#4DA6FF',
    'inflammation': '#FF7A78',
    'sunDamage': '#F2B56A',
    'elasticity': '#B68AFF',
}


def plot_label_distributions(labels: list[dict]):
    """Plot histograms and correlation matrix for teacher labels."""
    signals = ['structure', 'hydration', 'inflammation', 'sunDamage', 'elasticity']
    data = {sig: [l['signals'][sig] for l in labels] for sig in signals}
    df = pd.DataFrame(data)

    # Histograms
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    for ax, sig in zip(axes, signals):
        ax.hist(df[sig], bins=30, color=SIGNAL_COLORS[sig], alpha=0.8, edgecolor='white')
        ax.set_title(sig, fontsize=12, fontweight='bold')
        ax.set_xlabel('Score (0-100)')
        ax.set_ylabel('Count')
        ax.axvline(df[sig].mean(), color='black', linestyle='--', alpha=0.5)
    plt.suptitle(f'Teacher Label Distributions (n={len(labels)})', fontsize=14)
    plt.tight_layout()
    plt.savefig(str(LABELS_DIR / 'label_distributions.png'), dpi=150, bbox_inches='tight')
    plt.show()

    # Correlation matrix
    fig, ax = plt.subplots(1, 1, figsize=(7, 6))
    corr = df.corr()
    im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_xticks(range(len(signals)))
    ax.set_yticks(range(len(signals)))
    ax.set_xticklabels(signals, rotation=45, ha='right')
    ax.set_yticklabels(signals)
    for i in range(len(signals)):
        for j in range(len(signals)):
            ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=10)
    plt.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title('Signal Score Correlations')
    plt.tight_layout()
    plt.savefig(str(LABELS_DIR / 'signal_correlations.png'), dpi=150, bbox_inches='tight')
    plt.show()

    # Age vs elasticity (UTKFace subset)
    age_labels = [(l['age'], l['signals']['elasticity']) for l in labels if l.get('age')]
    if age_labels:
        ages, elast = zip(*age_labels)
        fig, ax = plt.subplots(1, 1, figsize=(8, 5))
        ax.scatter(ages, elast, alpha=0.3, s=10, c=SIGNAL_COLORS['elasticity'])
        z = np.polyfit(ages, elast, 2)  # Quadratic fit
        p = np.poly1d(z)
        x_range = np.linspace(min(ages), max(ages), 100)
        ax.plot(x_range, p(x_range), 'r--', linewidth=2, label='Quadratic fit')
        from scipy.stats import pearsonr
        r, _ = pearsonr(ages, elast)
        ax.set_xlabel('Age')
        ax.set_ylabel('Elasticity Score')
        ax.set_title(f'Age vs Teacher Elasticity Score (r={r:.3f})')
        ax.legend()
        plt.tight_layout()
        plt.savefig(str(LABELS_DIR / 'age_vs_elasticity.png'), dpi=150, bbox_inches='tight')
        plt.show()


if filtered_labels:
    plot_label_distributions(filtered_labels)
else:
    print('No labels to visualize yet.')

## Summary

After running this notebook you should have:
- `teacher_labels/labels_shard_XXXX.jsonl` — raw labels (checkpointed)
- `teacher_labels/train.json` — training set (~80%)
- `teacher_labels/val.json` — validation set (~10%)
- `teacher_labels/test.json` — test set (~10%)
- `teacher_labels/dataset_info.json` — metadata and statistics
- `teacher_labels/*.png` — distribution visualizations

**Next:** Use `11_distillation_training.ipynb` to train ONNX student models on these labels.